# 🦖 Botzilla: Enterprise Meeting Pipeline (Multi-Modal & Hinglish)
This notebook runs the advanced Botzilla pipeline on a T4 GPU. It supports:
1. **Audio-only**, **Unified Video**, or **Split Streams (Independent Audio & Video)**
2. **Hinglish Code-Switching** (via Whisper `initial_prompt` conditioning)
3. **Stable-State Video Extraction** (Advanced OpenCV Contour Differencing)
4. **Enterprise Azure JSON schemas** (Pydantic strictly-typed outputs)

**Instructions:** Configure Cell 2, upload your files in Cell 3, and run everything in order.

In [ ]:
# CELL 1: Install Dependencies
!pip install -q "numpy==2.0.2" "pandas==2.2.3" numba opencv-python-headless pydantic git+https://github.com/m-bain/whisperX.git

In [ ]:
# CELL 2: Configuration
import os

# --- HUGGINGFACE --- 
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    raise ValueError("❌ Please set your HuggingFace token!")

# --- PIPELINE MODE ---
# "UNIFIED" -> Upload 1 video file (audio is extracted from it)
# "AUDIO_ONLY" -> Upload 1 audio file (no video extraction)
# "SPLIT_STREAMS" -> Upload 1 video file AND 1 audio file (independent recordings)
PIPELINE_MODE = "UNIFIED"  

# If PIPELINE_MODE == "SPLIT_STREAMS", you can set an offset.
# e.g. If the video screen recording started 30.5 seconds AFTER the audio recording started:
VIDEO_OFFSET_SECONDS = 0.0  

# --- ASR SETTINGS ---
WHISPER_MODEL = "large-v3"
LANGUAGE = None # Auto-detect
BATCH_SIZE = 16
COMPUTE_TYPE = "float16"

# HINGLISH / CODE-SWITCHING CONDITIONING
# This prompt helps Whisper understand we expect mixed Hindi & English.
INITIAL_PROMPT = "The following is a corporate meeting with speakers using both English and Hindi (Hinglish). Technical terms are in English."

print(f"✅ Configured for {PIPELINE_MODE} mode!")


In [ ]:
# CELL 3: Upload Files
from google.colab import files
import os

VIDEO_FILE = None
AUDIO_FILE = None

if PIPELINE_MODE == "UNIFIED":
    print("📁 Upload your Unified Meeting Video (.mp4)...")
    uploaded = files.upload()
    VIDEO_FILE = list(uploaded.keys())[0]
    AUDIO_FILE = "extracted_audio.wav"
    print("🎵 Extracting audio using FFmpeg...")
    !ffmpeg -i "{VIDEO_FILE}" -q:a 0 -map a "{AUDIO_FILE}" -y -loglevel error
    print("✅ Unified Video ready!")

elif PIPELINE_MODE == "AUDIO_ONLY":
    print("📁 Upload your Audio File (.mp3, .wav)...")
    uploaded = files.upload()
    AUDIO_FILE = list(uploaded.keys())[0]
    print("✅ Audio Only ready!")

elif PIPELINE_MODE == "SPLIT_STREAMS":
    print("📁 First, upload your Independent AUDIO File (.mp3, .wav)...")
    uploaded_audio = files.upload()
    AUDIO_FILE = list(uploaded_audio.keys())[0]
    
    print("\n📁 Next, upload your Independent Screen Recording VIDEO File (.mp4)...")
    uploaded_video = files.upload()
    VIDEO_FILE = list(uploaded_video.keys())[0]
    print(f"\n✅ Split Streams ready! (Video Offset: {VIDEO_OFFSET_SECONDS}s)")


In [ ]:
# CELL 4: Video Slide Extraction (High-Precision Stable-State)
import cv2
import os

slides = []
SLIDES_DIR = "extracted_slides"

if PIPELINE_MODE in ["UNIFIED", "SPLIT_STREAMS"] and VIDEO_FILE:
    os.makedirs(SLIDES_DIR, exist_ok=True)
    print("🔄 Extracting key slides from video (High Precision)...")
    
    cap = cv2.VideoCapture(VIDEO_FILE)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps: fps = 30.0
    
    prev_frame = None
    frame_idx = 0
    
    # For "damn accurate" results, we sample 5 frames per second.
    # This catches very fast arrow movements or clicks.
    sample_rate = max(1, int(fps / 5)) 
    
    in_motion = False
    stable_frames_count = 0
    
    # Lower contour area threshold to catch smaller UI changes (like a mouse moving or arrow drawing)
    MIN_CONTOUR_AREA = 150  
    # Wait 1 full second (5 frames at 5 FPS) of ZERO motion before considering it "stable"
    STABILITY_THRESHOLD = 5 
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx % sample_rate == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (15, 15), 0) # Slightly lower blur to keep sharp small details
            
            if prev_frame is not None:
                frame_delta = cv2.absdiff(prev_frame, gray)
                thresh = cv2.threshold(frame_delta, 25, 255, cv2.THRESH_BINARY)[1]
                thresh = cv2.dilate(thresh, None, iterations=2)
                contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                significant_motion = any(cv2.contourArea(c) > MIN_CONTOUR_AREA for c in contours)
                
                timestamp_sec = frame_idx / fps
                
                if significant_motion:
                    in_motion = True
                    stable_frames_count = 0
                elif in_motion:
                    stable_frames_count += 1
                    if stable_frames_count >= STABILITY_THRESHOLD:
                        in_motion = False
                        
                        # Apply the offset to align this slide with the audio master timeline
                        aligned_sec = timestamp_sec + VIDEO_OFFSET_SECONDS
                        if aligned_sec < 0: aligned_sec = 0.0
                        
                        mm = int(aligned_sec // 60)
                        ss = int(aligned_sec % 60)
                        img_name = f"slide_{mm:02d}m{ss:02d}s.png"
                        img_path = os.path.join(SLIDES_DIR, img_name)
                        cv2.imwrite(img_path, frame)
                        
                        slides.append({
                            "timestamp_sec": round(aligned_sec, 2),
                            "timestamp_fmt": f"{mm:02d}:{ss:02d}",
                            "image_path": img_path
                        })
            else:
                # Capture the very first frame
                aligned_sec = 0.0 + VIDEO_OFFSET_SECONDS
                if aligned_sec < 0: aligned_sec = 0.0
                
                mm = int(aligned_sec // 60)
                ss = int(aligned_sec % 60)
                img_name = f"slide_initial_{mm:02d}m{ss:02d}s.png"
                img_path = os.path.join(SLIDES_DIR, img_name)
                cv2.imwrite(img_path, frame)
                slides.append({
                    "timestamp_sec": round(aligned_sec, 2),
                    "timestamp_fmt": f"{mm:02d}:{ss:02d}",
                    "image_path": img_path
                })
                    
            prev_frame = gray
        frame_idx += 1
    
    cap.release()
    print(f"✅ Extracted {len(slides)} highly accurate stable visual slides!")
else:
    print("⏭️ AUDIO_ONLY mode active. Skipping video slide extraction.")


In [ ]:
# CELL 5: Transcribe Audio (Anti-Hallucination + Hinglish)
import whisperx
import torch
import gc

device = "cuda"
# Pass Faster-Whisper kwargs via asr_options to eliminate hallucinations
asr_options = {
    "temperatures": [0.0],
    "condition_on_previous_text": False,
    "initial_prompt": INITIAL_PROMPT,
    "no_speech_threshold": 0.6,
    "log_prob_threshold": -1.0
}

print(f"🔄 Loading Whisper model '{WHISPER_MODEL}'...")
model = whisperx.load_model(WHISPER_MODEL, device, compute_type=COMPUTE_TYPE, language=LANGUAGE, asr_options=asr_options)
audio = whisperx.load_audio(AUDIO_FILE)

print("🔄 Transcribing (Hinglish + Anti-Hallucination Enabled)...")
transcribe_options = {
    "batch_size": BATCH_SIZE,
    "chunk_size": 30,
    "print_progress": True,
    "language": LANGUAGE,
}

result = model.transcribe(audio, **transcribe_options)

detected_lang = result.get("language", LANGUAGE)
print(f"\n✅ Transcription complete! Language: {detected_lang} | Segments: {len(result['segments'])}")
del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# CELL 6: Align Timestamps
print("🔄 Aligning timestamps...")
model_a, metadata = whisperx.load_align_model(language_code=detected_lang, device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
print("✅ Alignment complete!")
del model_a; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# CELL 7: Diarize (Pyannote)
print("🔄 Running Pyannote diarization...")
diarize_model = whisperx.diarize.DiarizationPipeline(token=HF_TOKEN, device=device)
diarize_segments = diarize_model(AUDIO_FILE)
result = whisperx.assign_word_speakers(diarize_segments, result)
print("✅ Diarization complete!")
del diarize_model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# CELL 8: Merge Visuals & Transcripts with Pydantic
from pydantic import BaseModel
from typing import List, Optional
import json

# Define strict Azure-ready output schemas
class SlideSnapshot(BaseModel):
    timestamp_sec: float
    timestamp_fmt: str
    image_path: str

class TranscriptSegment(BaseModel):
    speaker: str
    start: float
    end: float
    text: str
    slides_shown: List[str]

class BotzillaMetadata(BaseModel):
    pipeline_mode: str
    audio_file: str
    video_file: Optional[str]
    video_offset_seconds: float
    language: str
    num_slides_extracted: int

class BotzillaReport(BaseModel):
    metadata: BotzillaMetadata
    slides: List[SlideSnapshot]
    transcript: List[TranscriptSegment]

# Group and merge audio segments
merged_segments = []
for seg in result['segments']:
    spk = seg.get('speaker', 'UNKNOWN')
    if merged_segments and merged_segments[-1]['speaker'] == spk and seg['start'] - merged_segments[-1]['end'] < 2.0:
        merged_segments[-1]['end'] = seg['end']
        merged_segments[-1]['text'] += " " + seg['text'].strip()
    else:
        merged_segments.append({
            'speaker': spk, 
            'start': round(seg.get('start', 0.0), 2), 
            'end': round(seg.get('end', 0.0), 2), 
            'text': seg.get('text', '').strip()
        })

# Populate Pydantic models
pydantic_slides = [SlideSnapshot(**s) for s in slides]
pydantic_transcript = []

for seg in merged_segments:
    active_slides = [s.image_path for s in pydantic_slides if seg['start'] <= s.timestamp_sec <= seg['end']]
    
    pydantic_transcript.append(TranscriptSegment(
        speaker=seg['speaker'],
        start=seg['start'],
        end=seg['end'],
        text=seg['text'],
        slides_shown=active_slides
    ))

final_report = BotzillaReport(
    metadata=BotzillaMetadata(
        pipeline_mode=PIPELINE_MODE,
        audio_file=AUDIO_FILE,
        video_file=VIDEO_FILE,
        video_offset_seconds=VIDEO_OFFSET_SECONDS if PIPELINE_MODE == "SPLIT_STREAMS" else 0.0,
        language=detected_lang,
        num_slides_extracted=len(pydantic_slides)
    ),
    slides=pydantic_slides,
    transcript=pydantic_transcript
)

# Export strictly validated JSON
with open("botzilla_output.json", "w", encoding="utf-8") as f:
    if hasattr(final_report, 'model_dump'):
        f.write(final_report.model_dump_json(indent=2))
    else:
        f.write(final_report.json(indent=2))

print("✅ Exported botzilla_output.json with strictly-typed Pydantic schema!")

# Zipping everything for download
import shutil
if os.path.exists("extracted_slides"):
    !zip -r botzilla_results.zip botzilla_output.json extracted_slides/ > /dev/null
else:
    !zip botzilla_results.zip botzilla_output.json > /dev/null

from google.colab import files
files.download("botzilla_results.zip")
print("🎉 All done! Downloading results zip...")
